# 02 Data Cleaning and Preprocessing

This notebook performs the project’s explicit data cleaning and preprocessing step.

Cleaning decisions are documented here so the modeling workflow is auditable and not hidden inside later notebooks.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('d:/Banking and Finance/Projects/canadian-bank-loan-propensity-mlops-deployment')

In [2]:
import pandas as pd

from src.config import CLEANED_DATA_PATH, MERGED_DATA_PATH, RAW_DATA_1, RAW_DATA_2
from src.data_preprocessing import (
    get_data_quality_summary,
    load_dataset,
    merge_customer_datasets,
    clean_customer_data,
)

data1 = load_dataset(RAW_DATA_1)
data2 = load_dataset(RAW_DATA_2)

## 1. Merge Raw Customer Files

The raw files are merged using a one-to-one customer ID join. One-to-one validation prevents accidental row multiplication.

In [3]:
merged_data = merge_customer_datasets(data1, data2)

print(f"Merged data shape: {merged_data.shape}")
display(merged_data.head())

Merged data shape: (5000, 14)


,ID,Age,CustomerSince,HighestSpend,ZipCode,HiddenScore,MonthlyAverageSpend,Level,Mortgage,Security,FixedDepositAccount,InternetBanking,CreditCard,LoanOnCard
0,1,25,1,49,91107,4,1.6,1,0,1,0,0,0,NaN
1,2,45,19,34,90089,3,1.5,1,0,1,0,0,0,NaN
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0,NaN
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0,NaN
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,1,NaN


## 2. Pre-Cleaning Data Quality Review

In [4]:
display(get_data_quality_summary(merged_data))

,feature,data_type,non_null_count,missing_count,missing_pct,unique_values
0,ID,int64,5000,0,0.0,5000
1,Age,int64,5000,0,0.0,45
2,CustomerSince,int64,5000,0,0.0,47
3,HighestSpend,int64,5000,0,0.0,162
4,ZipCode,int64,5000,0,0.0,467
5,HiddenScore,int64,5000,0,0.0,4
6,MonthlyAverageSpend,float64,5000,0,0.0,108
7,Level,int64,5000,0,0.0,3
8,Mortgage,int64,5000,0,0.0,347
9,Security,int64,5000,0,0.0,2


In [5]:
negative_tenure_count = int((merged_data["CustomerSince"] < 0).sum())
missing_target_count = int(merged_data["LoanOnCard"].isna().sum())

print(f"Negative CustomerSince values: {negative_tenure_count}")
print(f"Missing target values: {missing_target_count}")

Negative CustomerSince values: 52
Missing target values: 20


## 3. Cleaning Decisions

- Remove rows with missing target values.
- Correct negative `CustomerSince` values to zero.
- Validate binary product flags.
- Validate duplicate rows and duplicate customer IDs.

In [6]:
cleaned_data = clean_customer_data(merged_data)

print(f"Cleaned data shape: {cleaned_data.shape}")
display(cleaned_data.head())

Cleaned data shape: (4980, 14)


,ID,Age,CustomerSince,HighestSpend,ZipCode,HiddenScore,MonthlyAverageSpend,Level,Mortgage,Security,FixedDepositAccount,InternetBanking,CreditCard,LoanOnCard
9,10,34,9,180,93023,1,8.9,3,0,0,0,0,0,1
10,11,65,39,105,94710,4,2.4,3,0,0,0,0,0,0
11,12,29,5,45,90277,3,0.1,2,0,0,0,1,0,0
12,13,48,23,114,93106,2,3.8,3,0,1,0,0,0,0
13,14,59,32,40,94920,4,2.5,2,0,0,0,1,0,0


## 4. Save Cleaned Data

The cleaned dataset is saved to `data/processed/`. This is a pipeline artifact, not a report.

In [7]:
MERGED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
merged_data.to_csv(MERGED_DATA_PATH, index=False)
cleaned_data.to_csv(CLEANED_DATA_PATH, index=False)

print(f"Merged data saved to: {MERGED_DATA_PATH}")
print(f"Cleaned data saved to: {CLEANED_DATA_PATH}")

Merged data saved to: D:\Banking and Finance\Projects\canadian-bank-loan-propensity-mlops-deployment\data\processed\01_merged_customer_data.csv
Cleaned data saved to: D:\Banking and Finance\Projects\canadian-bank-loan-propensity-mlops-deployment\data\processed\02_cleaned_customer_data.csv


In [8]:
display(get_data_quality_summary(cleaned_data))

,feature,data_type,non_null_count,missing_count,missing_pct,unique_values
0,ID,int64,4980,0,0.0,4980
1,Age,int64,4980,0,0.0,45
2,CustomerSince,int64,4980,0,0.0,44
3,HighestSpend,int64,4980,0,0.0,162
4,ZipCode,int64,4980,0,0.0,467
5,HiddenScore,int8,4980,0,0.0,4
6,MonthlyAverageSpend,float64,4980,0,0.0,108
7,Level,int8,4980,0,0.0,3
8,Mortgage,int64,4980,0,0.0,347
9,Security,int8,4980,0,0.0,2


## Cleaning Outcome

The dataset is now ready for EDA and feature engineering. The target is complete, tenure values are valid, binary columns are validated, and customer IDs remain unique.